In [ ]:
import fastf1 as ff
import numpy as np

# Stocker le cache
ff.Cache.enable_cache("donneescachef1")

# Charger la course spécifique (Bahrein, 2024) + données pilote
session = ff.get_session(2024, "Bahrein", "R")  
session.load()
laps_ver = session.laps.pick_driver("VER")
clean_laps = laps_ver.pick_quicklaps().pick_track_status("1") # On garde uniquement les tours sous drapeau vert 
chronos_secondes = clean_laps["LapTime"].dt.total_seconds().dropna() # Convertir en secondes pour une lecture + simple

# Calcul des paramètres de la loi Normale
mu_pilote = np.mean(chronos_secondes) # Moyenne des temps du pilote
sigma_pilote = np.std(chronos_secondes) # L'ecart type

print(f"Pilote : Max VERSTAPPEN (Bahrein, 2024)")
print(f"Nb de tours clean : {len(chronos_secondes)}")
print(f"Temps de base mu : {round(mu_pilote, 3)} sec")
print(f"Régularité sigma : {round(sigma_pilote, 3)} sec")


NotADirectoryError: Cache directory does not exist! Please check for typos or create it first.

In [ ]:
import pandas as pd 

# Préparation des variables t_base, Kfuel(N-L) et a(L) pour la régression

df_reg = pd.DataFrame() # On crée un tableau vide pour la régression

df_reg["LapTime_Y"] = clean_laps["LapTime"].dt.total_seconds() # Variable pour le chronos en secondes

df_reg["L"] = clean_laps["LapNumber"] # Variable de repère temporel L, le numéro de tours

N = 57 # Distance total de Bahrein (N) = 57 tours
df_reg["Fuel_Remaining_X1"] = N - df_reg['L'] # Variable pour le carburant restant (en tours)

df_reg["Tyre_Age_X2"] = clean_laps["TyreLife"] # Variable pour l'âge du pneu, récup dans fastf1 (en tours)

df_reg = df_reg.dropna() # removing missing value 

print(df_reg.head())

In [ ]:
import statsmodels.api as sm 

# On sépare les variables (X) explicatives de la cible (Y)
X = df_reg[["Fuel_Remaining_X1", "Tyre_Age_X2"]]
Y = df_reg["LapTime_Y"]

X = sm.add_constant(X) # On ajoute notre constante, t_base

# Création et résolution du modèle OLS
modele = sm.OLS(Y, X)
resultats = modele.fit()

print(resultats.summary()) # Affichage du rapport statistique complet 

# Exract propre pour le projet 
t_base = resultats.params["const"]
k_fuel = resultats.params["Fuel_Remaining_X1"]
rho = resultats.params["Tyre_Age_X2"]


print(f"t_base (Rythme absolu sans contrainte) : {round(t_base, 3)}s")
print(f"k_fuel (gain lié à l'essence/tour) : {round(k_fuel, 3)}s")
print(f"rho (Perte lié à la gomme par tour) : {round(rho, 3)}s")